# R1 — Access Control (Governance)

Row-level and column-level access control for the  lakehouse: adjuster teams get state-scoped access to `.gold._enriched` via a row filter, national ID / phone PII in `.raw.members` is masked behind a last-4 mask, `account users` receive narrowly scoped grants, and compliance gets proof tables plus a point-in-time grants snapshot in `gov`.

> Written per spec, not executed. Cells must be run top-to-bottom in order.

In [0]:
use catalog identifier(:catalog)

In [0]:

create schema if not exists gov;
create schema if not exists raw;
create or replace table raw.members as
select member_id, name, dob, gender, city, state, national_id, phone, email
from bronze.members;

create or replace table raw.plans as
select plan_code, plan_family, sum_insured, room_rent_cap_per_day, copay_pct,
       ped_waiting_months, initial_waiting_days, annual_premium
from bronze.plans;

create or replace table raw.hospitals as
select hospital_id, name, city, city_tier, state
from bronze.hospitals;

In [0]:


create or replace table gov.admin_users (
  user_name string
);

insert into gov.admin_users values
  ('deepaksinghal@cyntexa.com');              -- compliance admin / notebook owner

create or replace table gov.user_states (
  user_name string,
  state     string
);

insert into gov.user_states values
  ('deepaksinghal@cyntexa.com', 'Karnataka'),   -- TODO real adjuster account user
  ('adjuster.tg@example.com', 'Telangana');   -- TODO real adjuster account user

In [0]:


create or replace function gov.mask_last4(v string)
returns string
comment 'Column mask: admin_users see the full value; others see ****<last 4>.'
return case
         when v is null then null
         when exists (select 1
                      from gov.admin_users a
                      where lower(a.user_name) = lower(current_user()))
           then v
         else concat('****', right(v, 4))
       end;

alter table raw.members alter column national_id set mask gov.mask_last4;
alter table raw.members alter column phone      set mask gov.mask_last4;

In [0]:


create or replace function gov.claims_state_filter(p_hospital_id string)
returns boolean
comment 'Row filter: admins see all ; adjusters only hospitals in their mapped states; unmapped users see nothing.'
return
  exists (select 11
          from gov.admin_users a
          where lower(a.user_name) = lower(current_user()))
  or exists (
       select 1
       from gov.user_states us
       join raw.hospitals h
         on h.state = us.state
       where lower(us.user_name) = lower(current_user())
         and h.hospital_id = p_hospital_id
     );


alter table gold.claims_enriched
  set row filter gov.claims_state_filter on (hospital_id);



In [0]:


grant use catalog on catalog identifier(:catalog)  to `account users`;

grant use schema on schema gold to `account users`;
grant select     on schema gold to `account users`;

grant use schema on schema raw to `account users`;
grant select     on table raw.plans     to `account users`;
grant select     on table raw.hospitals to `account users`;

In [0]:
create or replace table gov.proof_members_admin as
select cast(member_id   as string) as member_id,
       cast(national_id as string) as national_id,
       cast(phone       as string) as phone
from raw.members
order by member_id
limit 10;

create or replace function gov.mask_adjusters_only(v string)
returns string
comment 'Column mask: users mapped in gov.user_states (adjusters) see ****<last 4>; everyone else sees the full value.'
return case
         when v is null then null
         when exists (select 1
                      from gov.user_states us
                      where lower(us.user_name) = lower(current_user()))
           then concat('****', right(v, 4))
         else v
       end;

-- Store full values; masking for user_states members is enforced at query time
create or replace table gov.proof_members_masked (
  member_id   string,
  national_id string,
  phone       string
);

insert into gov.proof_members_masked
select cast(member_id   as string),
       cast(national_id as string),
       cast(phone       as string)
from raw.members
order by member_id
limit 10;

alter table gov.proof_members_masked alter column national_id set mask gov.mask_adjusters_only;
alter table gov.proof_members_masked alter column phone      set mask gov.mask_adjusters_only;

select * from gov.proof_members_admin;
select * from gov.proof_members_masked;

In [0]:
create or replace table gov.grants_snapshot as
select cast(securable   as string) as securable,
       cast(principal   as string) as principal,
       cast(action_type as string) as action_type,
       cast(object_type as string) as object_type,
       cast(object_key  as string) as object_key
from (
  -- catalog-level privileges
  select catalog_name   as securable,
         grantee        as principal,
         privilege_type as action_type,
         'CATALOG'      as object_type,
         catalog_name   as object_key
  from information_schema.catalog_privileges

  union all

  -- schema-level privileges
  select concat(catalog_name, '.', schema_name) as securable,
         grantee                                as principal,
         privilege_type                         as action_type,
         'SCHEMA'                               as object_type,
         concat(catalog_name, '.', schema_name) as object_key
  from information_schema.schema_privileges

  union all

  -- table / view / MV / streaming-table privileges
  select concat(tp.table_catalog, '.', tp.table_schema, '.', tp.table_name)   as securable,
         tp.grantee                                                           as principal,
         tp.privilege_type                                                    as action_type,
         case t.table_type
           when 'MANAGED'           then 'TABLE'
           when 'EXTERNAL'          then 'TABLE'
           when 'VIEW'              then 'VIEW'
           when 'MATERIALIZED_VIEW' then 'MATERIALIZED_VIEW'
           when 'STREAMING_TABLE'   then 'STREAMING_TABLE'
           else t.table_type
         end                                                                  as object_type,
         concat(tp.table_catalog, '.', tp.table_schema, '.', tp.table_name)   as object_key
  from information_schema.table_privileges tp
  left join information_schema.tables t
         on  t.table_catalog = tp.table_catalog
         and t.table_schema  = tp.table_schema
         and t.table_name    = tp.table_name
);


select *
from gov.grants_snapshot
order by object_key, principal;

In [0]:

create or replace table gov.proof_claims_admin as
select cast(claim_id    as string) as claim_id,
       cast(hospital_id as string) as hospital_id
from gold.claims_enriched;

create or replace table gov.proof_claims_mapped as
select cast(c.claim_id    as string) as claim_id,
       cast(c.hospital_id as string) as hospital_id
from gold.claims_enriched c
join raw.hospitals h
  on h.hospital_id = c.hospital_id
where h.state in (select distinct state from gov.user_states);

create or replace table gov.proof_claims_unmapped as
select cast(claim_id    as string) as claim_id,
       cast(hospital_id as string) as hospital_id
from gold.claims_enriched
where 1 = 0;


select 'admin'     as persona, count(*) as visible_claims from gov.proof_claims_admin
union all
select 'mapped'    as persona, count(*) from gov.proof_claims_mapped
union all
select 'unmapped'  as persona, count(*) from gov.proof_claims_unmapped;

In [0]:


set tag on column raw.members.name        pii = true;
set tag on column raw.members.dob         pii = true;
set tag on column raw.members.national_id pii = true;
set tag on column raw.members.phone       pii = true;
set tag on column raw.members.email       pii = true;

In [0]:



comment on table gold.claims_enriched is
  'Member claim facts enriched to policy version at admission, plan attributes and hospital network status. Built by the claims_pipeline SDP pipeline; state-scoped by the claims.gov.claims_state_filter row filter on hospital_id.';
comment on table gold.hospital_monthly is
  'Monthly per-hospital claim aggregates (volume, billed, approved, rejected) derived from claims_enriched. Built by the claims_pipeline SDP pipeline.';
comment on table gold.hospital_monthly_designer is
  'Monthly per-hospital claim aggregates produced by the Lakeflow Designer prototype; must always match gold.hospital_monthly exactly.';

-- claims_enriched columns
comment on column gold.claims_enriched.claim_id              is 'Unique claim identifier (natural key from source claims feed).';
comment on column gold.claims_enriched.member_id             is 'Member identifier; join key to claims.raw.members (PII held there, not here).';
comment on column gold.claims_enriched.policy_id             is 'Policy identifier; resolved to the policy version effective at admission via claims.silver.policy_history.';
comment on column gold.claims_enriched.hospital_id           is 'Hospital identifier; keyed on by the claims.gov.claims_state_filter row filter.';
comment on column gold.claims_enriched.admission_date        is 'Admission date as shipped from source (string, not typed).';
comment on column gold.claims_enriched.discharge_date        is 'Discharge date as shipped from source (string, not typed); null while admitted.';
comment on column gold.claims_enriched.admission_month       is 'Admission month, DATE (first of month), derived with trunc(to_date(admission_date), MONTH).';
comment on column gold.claims_enriched.diagnosis_code        is 'Primary diagnosis code on the claim.';
comment on column gold.claims_enriched.procedure_code        is 'Primary procedure code on the claim.';
comment on column gold.claims_enriched.is_pre_existing       is 'Whether the condition is flagged pre-existing under the policy at admission.';
comment on column gold.claims_enriched.claim_type            is 'Claim category (e.g. inpatient/outpatient) from the source feed.';
comment on column gold.claims_enriched.billed_amount         is 'Amount billed by the hospital for the claim.';
comment on column gold.claims_enriched.room_rent_per_day     is 'Per-day room rent billed; compared against the plan cap for room_rent_over_cap.';
comment on column gold.claims_enriched.room_days             is 'Number of room-days billed on the claim.';
comment on column gold.claims_enriched.status                is 'Claim lifecycle status (submitted/approved/rejected/settled progression).';
comment on column gold.claims_enriched.approved_amount       is 'Amount approved by the insurer for the claim.';
comment on column gold.claims_enriched.rejection_code        is 'Rejection reason code; populated only for rejected claims.';
comment on column gold.claims_enriched.submitted_ts          is 'Timestamp the claim was submitted (string as shipped from source).';
comment on column gold.claims_enriched.settled_ts            is 'Timestamp the claim was settled (string as shipped from source); null until settled.';
comment on column gold.claims_enriched.channel               is 'Intake channel of the claim (optional source column, not present in every batch file).';
comment on column gold.claims_enriched.plan_code_at_admission is 'Plan code from the policy version effective at admission (temporal join to policy_history).';
comment on column gold.claims_enriched.plan_family           is 'Plan family (gold/silver/corporate/senior) for the plan effective at admission.';
comment on column gold.claims_enriched.room_rent_cap_per_day is 'Per-day room rent cap of the plan effective at admission (string from plans feed).';
comment on column gold.claims_enriched.policy_start_date     is 'Earliest recorded effective date of the policy (min __START_AT from policy_history).';
comment on column gold.claims_enriched.policy_tenure_months  is 'Months between policy start and admission (integer months_between).';
comment on column gold.claims_enriched.length_of_stay        is 'Length of stay in days = datediff(discharge_date, admission_date); null while not discharged.';
comment on column gold.claims_enriched.turnaround_days       is 'Days from submitted_ts to settled_ts; null unless the claim is settled.';
comment on column gold.claims_enriched.room_rent_over_cap    is 'True when room_rent_per_day exceeds the plan cap (try_cast to double); null when cap is not numeric.';
comment on column gold.claims_enriched.is_network_at_admission is 'True when the hospital was in the insurer network on the admission date (never null; false when out of network).';
comment on column gold.claims_enriched.hospital_city         is 'City of the treating hospital (from raw.hospitals).';
comment on column gold.claims_enriched.hospital_city_tier    is 'City tier of the treating hospital (from raw.hospitals).';

-- hospital_monthly columns
comment on column gold.hospital_monthly.hospital_id     is 'Hospital identifier; one row per hospital x admission month.';
comment on column gold.hospital_monthly.admission_month is 'Admission month (DATE, first of month) the aggregate covers.';
comment on column gold.hospital_monthly.claim_count     is 'Number of enriched claims for the hospital-month (int).';
comment on column gold.hospital_monthly.total_billed    is 'Sum of billed_amount for the hospital-month (decimal 18,2).';
comment on column gold.hospital_monthly.total_approved  is 'Sum of approved_amount, coalesced to 0 — never empty (decimal 18,2).';
comment on column gold.hospital_monthly.rejected_count  is 'Number of claims with status rejected for the hospital-month (int).';

-- hospital_monthly_designer columns
comment on column gold.hospital_monthly_designer.hospital_id     is 'Hospital identifier; one row per hospital x admission month (Designer prototype).';
comment on column gold.hospital_monthly_designer.admission_month is 'Admission month (DATE, first of month) the aggregate covers (Designer prototype).';
comment on column gold.hospital_monthly_designer.claim_count     is 'Number of enriched claims for the hospital-month (int; must match hospital_monthly).';
comment on column gold.hospital_monthly_designer.total_billed    is 'Sum of billed_amount for the hospital-month (decimal 18,2; must match hospital_monthly).';
comment on column gold.hospital_monthly_designer.total_approved  is 'Sum of approved_amount, coalesced to 0 (decimal 18,2; must match hospital_monthly).';
comment on column gold.hospital_monthly_designer.rejected_count  is 'Number of rejected claims for the hospital-month (int; must match hospital_monthly).';

In [0]:
create or replace table gov.audit_claims_enriched as
select cast(version    as bigint)    as version,
       cast(changed_at as timestamp) as changed_at,
       cast(changed_by as string)    as changed_by,
       cast(operation  as string)    as operation,
       cast(source     as string)    as source
from (
  select * from (
    select cast(null as bigint)  as version,
           event_time            as changed_at,
           user_identity.email   as changed_by,
           action_name           as operation,
           'system.access.audit' as source
    from system.access.audit
    where service_name = 'unityCatalog'
      and action_name  = 'updateTables'
      and request_params['full_name_arg'] = concat(:catalog, '.gold.claims_enriched')
    order by event_time desc
    limit 2
  )
  union all
  select * from (
    select version,
           timestamp        as changed_at,
           userName         as changed_by,
           operation,
           'describe_history' as source
    from (describe history gold.claims_enriched)
    order by version desc
    limit 1
  )
);

select * from gov.audit_claims_enriched order by changed_at desc;